# Sesión 2 — Consulta, Filtrado e Indexación en Pandas

**Curso:** Especialización en IA y Big Data / Programación en IA  
**Duración:** 1 Hora  
**Objetivos:**
- Dominar la proyección y selección de columnas devolviendo estructuras `pd.Series` y `pd.DataFrame`.
- Aprender a renombrar atributos y manipular el índice (`set_index`).
- Dominar la selección por posición numérica con `.iloc` y por etiquetas o condiciones con `.loc`.
- Aplicar indexación booleana combinando operadores lógicos (`&`, `|`, `~`), `.isin()` y `.str.contains()`.
- Utilizar la sintaxis declarativa de consultas avanzadas mediante el método `.query()`.

--- 
## 0. Preparación del Entorno y Carga del Dataset

Comenzaremos importando la pila básica e importando el dataset de ventas de videojuegos (**Video Game Sales**) directamente desde URL pública.

In [ ]:
import numpy as np
import pandas as pd

# Carga del dataset de ventas de videojuegos para la sesión
url_vgsales = 'https://raw.githubusercontent.com/selva86/datasets/master/vgsales.csv'
df = pd.read_csv(url_vgsales)

# Inspección rápida de las primeras filas
display(df.head())
print(f"Dimensiones iniciales del dataset: {df.shape}")

--- 
## Bloque 1: Selección de Series y DataFrames, Renombrado e Índices (15 min)

En Pandas, seleccionar una columna o varias cambia la estructura del objeto devuelto (`pd.Series` vs. `pd.DataFrame`).

### 1.1 Proyección de columnas: `pd.Series` vs `pd.DataFrame`

In [ ]:
# 1. Un solo par de corchetes -> Devuelve una pd.Series (1D)
serie_nombres = df['Name']
print("Tipo al usar df['Name']:", type(serie_nombres))
display(serie_nombres.head(3))

print("-" * 50)

# 2. Doble par de corchetes -> Devuelve un pd.DataFrame (2D) incluso con 1 sola columna
df_nombres = df[['Name']]
print("Tipo al usar df[['Name']]:", type(df_nombres))
display(df_nombres.head(3))

print("-" * 50)

# 3. Selección de múltiples columnas -> Devuelve un pd.DataFrame (2D)
df_subconjunto = df[['Name', 'Platform', 'Global_Sales']]
display(df_subconjunto.head(3))

### 1.2 Renombrado de columnas y reajuste de índices (`set_index`)

In [ ]:
# Renombrar columnas explícitamente mediante un diccionario con .rename()
df_renombrado = df.rename(columns={
    'Name': 'Nombre_Juego',
    'Platform': 'Plataforma',
    'Global_Sales': 'Ventas_Globales_M'
})
display(df_renombrado[['Nombre_Juego', 'Plataforma', 'Ventas_Globales_M']].head(3))

# Establecer una columna existente como nuevo índice del DataFrame (ejemplo: 'Rank')
df_indexed = df.set_index('Rank')
display(df_indexed.head(3))

--- 
## Bloque 2: Indexación por Posición (`.iloc`) y por Etiquetas (`.loc`) (25 min)

Para acceder a secciones específicas de un DataFrame utilizaremos los indexadores `.iloc` (basado en posiciones enteras) y `.loc` (basado en nombres de etiquetas o condiciones booleanas).

### 2.1 Indexación Posicional: `.iloc[filas, columnas]`
Utiliza posiciones numéricas enteras basadas en índice 0 (estilo *slicing* de Python: `inicio:fin_excluido`).

In [ ]:
# Primeras 5 filas (0 a 4) y columnas de la posición 1 a la 4 (excluyendo la 4)
display(df.iloc[0:5, 1:4])

# Filas específicas arbitrarias [0, 10, 20] y columnas [1, 2, 10]
display(df.iloc[[0, 10, 20], [1, 2, 10]])

### 2.2 Indexación por Etiquetas y Filtrado Booleano: `.loc[filas, columnas]`
Acepta nombres de etiquetas de columnas y máscaras condicionales.

In [ ]:
# Slicing de columnas por sus nombres de etiqueta (a diferencia de iloc, loc SÍ incluye el final)
display(df.loc[0:4, 'Name':'Genre'])

# Indexación Booleana simple: Filtrar juegos con ventas globales mayores a 20 millones
mascara_ventas = df['Global_Sales'] > 20.0
display(df.loc[mascara_ventas, ['Name', 'Platform', 'Global_Sales']])

### 2.3 Combinación de Condiciones Lógicas (`&`, `|`, `~`)
En Pandas **no** se usan los operadores `and`, `or`, `not` de Python para evaluar Series, sino los operadores vectoriales a nivel de bits:
- `&` -> AND (Ambas condiciones se cumplen)
- `|` -> OR (Al menos una condición se cumple)
- `~` -> NOT (Negación / Inverso)

> **Regla fundamental:** Cada condición individual DEBE ir envuelta estrictamente entre paréntesis `()`.

In [ ]:
# Filtrar juegos de Acción (Action) lanzados después del año 2015
cond_accion = df['Genre'] == 'Action'
cond_reciente = df['Year'] > 2015

df_accion_reciente = df.loc[(cond_accion) & (cond_reciente), ['Name', 'Platform', 'Year', 'Genre', 'Global_Sales']]
display(df_accion_reciente.head())

### 2.4 Filtrado Categórico y de Texto (`.isin()` y `.str.contains()`)

In [ ]:
# 1. Uso de .isin() para filtrar pertenencia a una lista de categorías (ej: Nintendo 64 o Game Boy)
plataformas_retro = ['N64', 'GB']
df_retro = df.loc[df['Platform'].isin(plataformas_retro), ['Name', 'Platform', 'Year', 'Global_Sales']]
display(df_retro.head(3))

print("-" * 50)

# 2. Búsqueda de patrones de texto en columnas de cadenas con .str.contains()
# Buscamos títulos de juegos que contengan la palabra 'Mario' (case=False ignora mayúsculas/minúsculas)
df_mario = df.loc[df['Name'].str.contains('Mario', case=False, na=False), ['Name', 'Platform', 'Global_Sales']]
display(df_mario.head(3))

--- 
## Bloque 3: Consultas Avanzadas con el Método `.query()` (10 min)

El método `.query()` permite realizar filtros sobre DataFrames complejos utilizando una sintaxis declarativa más limpia en forma de cadena de texto.

In [ ]:
# Sintaxis de query(): las variables de columna se escriben directamente sin repetición de df['...']
# Consulta: Ventas Globales > 10.0 M Y Plataforma en PS3 o PS4
query_str = 'Global_Sales > 10.0 and Platform in ["PS3", "PS4"]'
df_query_res = df.query(query_str)
display(df_query_res[['Name', 'Platform', 'Year', 'Global_Sales']])

# Referenciar variables de Python dentro de query() usando el prefijo '@'
min_ventas = 15.0
genero_búsqueda = 'Shooter'
df_var_query = df.query('Global_Sales >= @min_ventas and Genre == @genero_búsqueda')
display(df_var_query[['Name', 'Platform', 'Genre', 'Global_Sales']])

--- 
## Bloque 4: Puesta en Práctica Guiada (Mini-ejercicio) (10 min)

Resolveremos conjuntamente un caso práctico combinando filtros cuantitativos y cualitativos.

In [ ]:
# MINI-EJERCICIO GUIADO:
# Encontrar los juegos publicados por 'Electronic Arts' o 'Nintendo',
# de género 'Role-Playing' o 'Sports',
# lanzados entre los años 2000 y 2010 (ambos inclusive),
# que hayan vendido más de 5 millones en Norteamérica (NA_Sales).

filtro_guiado = (
    (df['Publisher'].isin(['Electronic Arts', 'Nintendo'])) &
    (df['Genre'].isin(['Role-Playing', 'Sports'])) &
    (df['Year'] >= 2000) & (df['Year'] <= 2010) &
    (df['NA_Sales'] > 5.0)
)

df_resultado_guiado = df.loc[filtro_guiado, ['Name', 'Platform', 'Year', 'Genre', 'Publisher', 'NA_Sales']]
display(df_resultado_guiado)

--- 
## 🏋️‍♂️ Ejercicios de Práctica Autónoma para los Alumnos

Realiza los siguientes 4 ejercicios utilizando el DataFrame `df` de ventas de videojuegos cargado al inicio.

### Ejercicio 1: Selección y Proyección Específica
1. Selecciona del DataFrame `df` únicamente las columnas `Name`, `Platform` y `Global_Sales` devolviendo una estructura `pd.DataFrame`.
2. Muestra mediante `.iloc` los registros situados entre las filas **50 y 60** (ambas inclusive en posición) para las 3 columnas seleccionadas.
3. Renombra temporalmente la columna `Global_Sales` como `Ventas_Totales`.

In [ ]:
# Escribe aquí tu solución para el Ejercicio 1:


### Ejercicio 2: Indexación Booleana Compuesta con Operadores Lógicos
1. Filtra los juegos pertenecientes a la plataforma **'PS4'** o **'XOne'**.
2. Añade la condición de que el año de lanzamiento (`Year`) sea estrictamente **posterior a 2014**.
3. Incluye el requisito de que las ventas en Europa (`EU_Sales`) sean superiores a **2.0 millones de copias**.
4. Muestra únicamente las columnas `Name`, `Platform`, `Year` y `EU_Sales` utilizando `.loc`.

In [ ]:
# Escribe aquí tu solución para el Ejercicio 2:


### Ejercicio 3: Búsqueda de Patrones de Texto y Listas de Categorías
1. Utiliza `.str.contains()` para encontrar todos los títulos de juegos cuyo nombre contenga la palabra **'Pokemon'** o **'Zelda'** (sin diferenciar mayúsculas de minúsculas).
2. Asegúrate de filtrar únicamente los juegos de distribuidores (`Publisher`) que estén en la lista `['Nintendo', 'Capcom']` utilizando `.isin()`.
3. Muestra el número total de juegos encontrados (`len` o `.shape`) y despliega las primeras 10 filas ordenadas por ventas globales.

In [ ]:
# Escribe aquí tu solución para el Ejercicio 3:


### Ejercicio 4: Consultas Expresivas Avanzadas con `.query()`
1. Reescribe la siguiente consulta utilizando **exclusivamente el método `.query()`**:
   - Juegos del género **'Racing'** o **'Action'**.
   - Ventas en Japón (`JP_Sales`) superiores a **1.0 millón de copias**.
   - Año de publicación anterior al **2005**.
2. Almacena el resultado en la variable `df_carreras_accion` y muestra sus dimensiones e inspecciona las primeras filas.

In [ ]:
# Escribe aquí tu solución para el Ejercicio 4:
